# ICEEICT 2027 — M3 fine-tune (XLM-R) and GPU evaluation

Requires the Kaggle Dataset `bnuni-bundle` (the zip made by `tools/make_kaggle_bundle.py`) attached to this notebook, and a GPU (T4 x2 or P100).

Run all cells. Output: `/kaggle/working/bnuni_results.zip` containing `models/` logs and `results/`.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile
# Kaggle usually UNZIPS an uploaded archive, so the bundle may appear either as the zip itself
# or as an extracted tree (a folder containing src/, data/, fonts/, SSOT.md). Accept both.
PROJ = '/kaggle/working/proj'
shutil.rmtree(PROJ, ignore_errors=True); os.makedirs(PROJ)
zips = glob.glob('/kaggle/input/**/bnuni_bundle.zip', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p))) for p in glob.glob('/kaggle/input/**/src/eval/run_eval.py', recursive=True)]
if zips:
    zipfile.ZipFile(zips[0]).extractall(PROJ); print('extracted', zips[0])
elif trees:
    src_root = trees[0]
    for item in ('src', 'data', 'fonts', 'SSOT.md'):
        s = os.path.join(src_root, item)
        if os.path.isdir(s): shutil.copytree(s, os.path.join(PROJ, item))
        elif os.path.isfile(s): shutil.copy(s, PROJ)
    print('copied extracted bundle from', src_root)
else:
    print('Contents of /kaggle/input:'); [print(' ', p) for p in glob.glob('/kaggle/input/**', recursive=True)[:60]]
    raise SystemExit('bnuni bundle not found: attach the bnuni-bundle dataset under Input (right panel)')
os.chdir(PROJ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'protobuf'], check=False)
import torch, transformers; print('torch', torch.__version__, 'transformers', transformers.__version__, 'cuda', torch.cuda.is_available())
print(sorted(os.listdir(PROJ)))

In [ ]:
# Fine-tune M3 = xlm-roberta-base on BD-SHS train, three seeds (about 10-12 min each on a T4)
for seed in (1, 2, 3):
    out = f'/kaggle/working/models/m3_seed{seed}'
    if os.path.exists(out + '/train_log.json'):
        print('exists', out); continue
    subprocess.run([sys.executable, 'src/train/finetune_xlmr.py', '--seed', str(seed), '--out', out,
                    '--epochs', '3', '--lr', '2e-5', '--bs', '32', '--max-length', '128'], check=True)

In [ ]:
# Evaluation on GPU: pilot (val, 200+200) and final (test, all eligible rows).
# Model spec = path_or_hub_id:harmful_index. Hub models are included so every model is scored by the same code on the same machine.
MODELS = ['/kaggle/working/models/m3_seed1:1', '/kaggle/working/models/m3_seed2:1', '/kaggle/working/models/m3_seed3:1',
          'Hate-speech-CNERG/bengali-abusive-MuRIL:1', 'tajuarAkash/enhanced-banglabert-hate-speech:1']
common = ['--families', 'F1', 'F2', 'F2nbsp', 'F5', '--budgets', '1', '3', 'len', '--bs', '128']
subprocess.run([sys.executable, 'src/eval/run_eval.py', '--split', 'val', '--n-per-class', '200', '--out', '/kaggle/working/results/pilot', '--models', *MODELS, *common], check=True)
subprocess.run([sys.executable, 'src/eval/run_eval.py', '--split', 'test', '--out', '/kaggle/working/results/final', '--models', *MODELS, *common], check=True)

In [ ]:
# Statistics (same script as local) and packaging
subprocess.run([sys.executable, 'src/stats/analyze.py', '--dir', '/kaggle/working/results/pilot', '--split', 'val'], check=False)
subprocess.run([sys.executable, 'src/stats/analyze.py', '--dir', '/kaggle/working/results/final', '--split', 'test'], check=False)
with zipfile.ZipFile('/kaggle/working/bnuni_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk('/kaggle/working/results'):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), '/kaggle/working'))
    for seed in (1, 2, 3):
        p = f'/kaggle/working/models/m3_seed{seed}/train_log.json'
        if os.path.exists(p): z.write(p, os.path.relpath(p, '/kaggle/working'))
print('done: /kaggle/working/bnuni_results.zip', os.path.getsize('/kaggle/working/bnuni_results.zip') // 1024, 'KB')